# Python Write Files

> 📘 **Python Mastery** · Module 06 — File Handling · Lesson 2/4

Reading files gave your program a memory; **writing** lets it leave a record — scores, reports, logs, results. One warning before we start: write mode is the only beginner tool that can *silently destroy hours of data*. Learn its temper today and it will never bite you.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- Open a file in `"w"` mode and describe exactly what happens to existing content.
- Predict (and avoid) the silent-overwrite trap that deletes files without asking.
- Use `.write()` and interpret the character count it returns.
- Convert numbers with `str()` / f-strings before writing them.
- Explain why `writelines()` does **not** insert newlines — and fix it.
- Build a formatted report file from Python data, always inside a `with` block.

## 1. Writing with `"w"` Mode

Writing is the mirror of reading: same `open()`, different intention in the mode string. `"w"` opens a file for **writing from the top**. If the file does not exist, Python creates it; if it *does* exist, Python empties it first. Either way you begin with a blank page.

**Syntax:**

```python
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("some text")        # no \n is added for you!
```

**Example:** we write a file, then read it back immediately to prove what landed on disk.

In [1]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

with open("sample_data/letter.txt", "w", encoding="utf-8") as f:
    f.write("Dear Sarah,\n")
    f.write("Writing files is easier than it sounds.\n")
    f.write("— Your future self\n")

# Read it back to confirm what actually reached the disk.
print(Path("sample_data/letter.txt").read_text(encoding="utf-8"))

Dear Sarah,
Writing files is easier than it sounds.
— Your future self



## 2. The Silent Overwrite Trap

Here is the danger promised in the intro. `"w"` does not ask "are you sure?". The instant `open()` succeeds, your old file is emptied — *before you have written anything*. If your program then crashes, you are left with an empty file where your data used to be.

**Rule of thumb:** `"w"` means *"replace the whole file"*. If you mean *"add to the file"*, that is `"a"` (next lesson). If you mean *"never destroy what is there"*, that is `"x"` (also next lesson).

**Example:** watch Version 1 disappear.

In [2]:
path = "sample_data/danger.txt"

# Write #1: precious research results.
with open(path, "w", encoding="utf-8") as f:
    f.write("Version 1: three months of experiments\n")
print("After write #1:", repr(Path(path).read_text(encoding="utf-8")))

# Write #2: same mode, innocent-looking... and Version 1 is GONE.
with open(path, "w", encoding="utf-8") as f:
    f.write("Version 2\n")
print("After write #2:", repr(Path(path).read_text(encoding="utf-8")))

After write #1: 'Version 1: three months of experiments\n'
After write #2: 'Version 2\n'


## 3. `.write()` Returns a Count

`.write()` is not silent — it returns the **number of characters** actually written. Beginners ignore this value; it is quietly useful for progress logs, verifying that a big serialisation completed, or building counters.

**Syntax:**

```python
count = f.write(text)     # count == len(text), in characters
```

**Example:**

In [3]:
with open("sample_data/counted.txt", "w", encoding="utf-8") as f:
    first  = f.write("Python Mastery")
    second = f.write(" makes files easy!")
    third  = f.write("\n")

print(f"wrote {first} + {second} + {third} characters")
print("len() agrees:", first + second + third ==
      len(Path("sample_data/counted.txt").read_text(encoding="utf-8")))

wrote 14 + 18 + 1 characters
len() agrees: True


## 4. Numbers Must Become Strings

Files in text mode store **characters**. Hand `.write()` a number and it refuses loudly with `TypeError` — Python will not guess whether `65` should become `"65"`, `"A"`, or `"65.0"`. Convert explicitly: `str(value)`, or better, an f-string that formats *and* converts in one step.

**Syntax:**

```python
f.write(str(42))            # explicit conversion
f.write(f"score: {42}")     # f-strings convert for you
f.write(f"pi ~ {3.14159:.2f}")   # ...and format decimals
```

**Example:**

In [4]:
score = 97
bonus = 0.256

# The refusal:
try:
    with open("sample_data/rejected.txt", "w", encoding="utf-8") as f:
        f.write(score)                      # int, not str!
except TypeError as e:
    print("Rejected ->", e)

# The fixes:
with open("sample_data/accepted.txt", "w", encoding="utf-8") as f:
    f.write("Score: " + str(score) + "\n")          # fix 1: str()
    f.write(f"Bonus rate: {bonus:.1%}\n")           # fix 2: f-string

print(Path("sample_data/accepted.txt").read_text(encoding="utf-8"))

Rejected -> write() argument must be str, not int
Score: 97
Bonus rate: 25.6%



## 5. Newlines Are Your Job

Unlike `print()`, `.write()` adds **nothing** between calls — not even a space. Whatever line structure you want on disk, you must spell out with `\n` yourself. Forgetting it is how files end up as one enormous unreadable line.

**Syntax:**

```python
f.write("first line\n")     # \n ends the record
f.write("second line\n")
```

**Example:**

In [5]:
shopping = ["milk", "eggs", "rice"]

# Forgot the newlines -> everything glued into one token.
with open("sample_data/glued_shopping.txt", "w", encoding="utf-8") as f:
    for item in shopping:
        f.write(item)
print("glued:", repr(Path("sample_data/glued_shopping.txt").read_text(encoding="utf-8")))

# Added \n -> one proper line per item.
with open("sample_data/tidy_shopping.txt", "w", encoding="utf-8") as f:
    for item in shopping:
        f.write(item + "\n")
print("tidy :", repr(Path("sample_data/tidy_shopping.txt").read_text(encoding="utf-8")))

glued: 'milkeggsrice'
tidy : 'milk\neggs\nrice\n'


In [6]:
# Multi-line content in one call: a triple-quoted string already
# CONTAINS its newlines, so a single .write() is enough.
menu = """Dhaka Diner — Today's Menu
--------------------------------
Kacchi Biryani ... 320 BDT
Fuchka ...........  60 BDT
Mango Lassi ......  90 BDT
"""

with open("sample_data/menu.txt", "w", encoding="utf-8") as f:
    f.write(menu)

print(Path("sample_data/menu.txt").read_text(encoding="utf-8"))

Dhaka Diner — Today's Menu
--------------------------------
Kacchi Biryani ... 320 BDT
Fuchka ...........  60 BDT
Mango Lassi ......  90 BDT



## 6. `writelines()` — the Misleading Name

`f.writelines(list_of_strings)` writes every string in the iterable, back to back. Despite the name — inherited from decades of C tradition — it inserts **no newlines whatsoever**. Treat it as `write_each()` : you supply the `\n` inside the items yourself. Its advantage over a loop is speed and brevity when your strings are already line-shaped.

**Syntax:**

```python
f.writelines(["a\n", "b\n"])          # correct: newlines included by you
f.writelines(name + "\n" for name in names)   # works with generators too
```

**Example:**

In [7]:
players = ["Sarah", "Tanvir", "Amina", "Rafi"]

# Gotcha reproduced: writelines() adds NOTHING between items.
with open("sample_data/glued_players.txt", "w", encoding="utf-8") as f:
    f.writelines(players)
print("glued:", repr(Path("sample_data/glued_players.txt").read_text(encoding="utf-8")))

# Fixed: attach \n to every item first.
with open("sample_data/tidy_players.txt", "w", encoding="utf-8") as f:
    f.writelines(p + "\n" for p in players)
print("tidy :", repr(Path("sample_data/tidy_players.txt").read_text(encoding="utf-8")))

glued: 'SarahTanvirAminaRafi'
tidy : 'Sarah\nTanvir\nAmina\nRafi\n'


## 7. `"w"` Creates Missing Files

One friendly property: `"w"` never complains about a missing file — it simply creates it. This is how programs produce brand-new outputs (exports, saves, reports) without any setup. Combined with Section 2 it cuts both ways: creation for free, destruction for free.

**Syntax:**

```python
with open("brand_new_file.txt", "w", encoding="utf-8") as f:   # born here
    f.write("hello, world\n")
```

**Example:**

In [8]:
from pathlib import Path

fresh = Path("sample_data", "brand_new.txt")
print("Existed before?", fresh.exists())          # False

with open(fresh, "w", encoding="utf-8") as f:     # ...born right here
    f.write("Born just now, thanks to 'w'.\n")

print("Existed after ?", fresh.exists())
print(fresh.read_text(encoding="utf-8"))

Existed before? False
Existed after ? True
Born just now, thanks to 'w'.



## 8. Mini-Project: Saving a Structured Report

Real programs rarely write random sentences — they format **structured data** into readable documents. The recipe: build the lines as a list of strings (easy to test, easy to reorder), `"\n".join()` them, write once, done. Here Sarah's quiz scores become a ranked report card.

In [9]:
from pathlib import Path

scores = {"Sarah": 92, "Tanvir": 78, "Amina": 95, "Rafi": 66}
average = sum(scores.values()) / len(scores)

lines = []
lines.append("PYTHON QUIZ - SCORE REPORT")
lines.append("=" * 30)
for name, score in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
    bar = "#" * (score // 10)                       # tiny text chart
    marker = " <-- top!" if score == max(scores.values()) else ""
    lines.append(f"{name:<8} {score:>3}  {bar}{marker}")
lines.append("-" * 30)
lines.append(f"Class average : {average:.1f}")
lines.append(f"Generated for : Sarah's study group")

report = "\n".join(lines) + "\n"

with open("sample_data/report.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Verify the artefact exactly as a colleague would see it.
print(Path("sample_data/report.txt").read_text(encoding="utf-8"))

PYTHON QUIZ - SCORE REPORT
Amina     95  ######### <-- top!
Sarah     92  #########
Tanvir    78  #######
Rafi      66  ######
------------------------------
Class average : 82.8
Generated for : Sarah's study group



## 9. Always Write Inside `with`

Writing amplifies the cost of a leaky file handle. Drop that closing step and your last lines can vanish even though "the program wrote them". The `with` statement flushes and closes on every exit path, including exceptions. There is no excuse left: **every** write in this course lives inside a `with`.

> 🔍 **Under the Hood:** `.write()` does not touch your disk on every call. Text lands in a small in-memory buffer; Python pushes it to the operating system only when the buffer fills — or when the file closes. An unclosed file can therefore hold your newest data hostage in RAM, invisible to other programs and lost to a crash. Closing is not tidiness, it is the commit button.

**Syntax:**

```python
with open(path, "w", encoding="utf-8") as out:    # save
    out.write(data)
with open(path, encoding="utf-8") as inp:         # load right back
    same_data = inp.read()
```

**Example:** the classic save-then-load round-trip.

In [10]:
results = [88, 92, 79, 95]

# SAVE: one number per line, so reading back is trivial.
with open("sample_data/results.txt", "w", encoding="utf-8") as out:
    for r in results:
        out.write(f"{r}\n")

# LOAD: reconstruct the numbers from disk.
loaded = []
with open("sample_data/results.txt", encoding="utf-8") as inp:
    for line in inp:
        loaded.append(int(line.strip()))

print("saved :", results)
print("loaded:", loaded, "-> total", sum(loaded))

saved : [88, 92, 79, 95]
loaded: [88, 92, 79, 95] -> total 354


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| Opening an existing file with `"w"` to "just add something" | Entire file wiped, instantly and silently | Use `"a"` to extend, `"x"` to refuse overwriting |
| `f.write(42)` | `TypeError` — text mode stores characters only | `f.write(str(42))` or `f.write(f"{42}")` |
| `f.writelines(items)` expecting one-per-line | One giant concatenated line | Attach `"\n"` to each item yourself |
| Forgetting `"\n"` between `.write()` calls | Output glued together, unreadable | End every logical record with `"\n"` |
| Testing writes with `print(f.write(x))` alone | Looks fine in console; buffer may not be flushed | Close via `with`, then read the file back |
| No `encoding=` on writes | Files written in a machine-specific encoding break elsewhere | Always `encoding="utf-8"` |

## 💡 Best Practices & Pro Tips

- Pause before every `"w"` and ask: *may this file be destroyed?* If not, use `"a"` or `"x"`.
- Compose documents as a `list` of strings and `"\n".join(...)` once — cleaner than dozens of `.write()` calls and trivially unit-testable.
- Format numbers with f-string specs (`{avg:.1f}`, `{pct:.0%}`) so files are readable by humans *and* parseable by machines.
- After important writes, read the file back (or check `Path(...).stat().st_size > 0`) before declaring victory.
- Keep filenames descriptive and lowercase-with-underscores: `report_2026_08_26.txt` sorts and greps beautifully.
- **AI-engineering relevance:** training scripts constantly *write*: metric logs per epoch, prediction dumps, tokenised dataset shards, experiment configs. They are all `with open(..., "w")` underneath — and the silent-overwrite trap is precisely how teams have lost fine-tuned checkpoints by pointing two jobs at the same filename.

## 📌 Summary

| Tool | What it does | Example |
|------|--------------|---------|
| `open(p, "w", encoding=...)` | Create-or-wipe, then write from the top | `open("r.txt", "w", encoding="utf-8")` |
| `.write(s)` | Write a string; returns char count | `n = f.write("hi\n")` |
| `.writelines(iterable)` | Write many strings — **no newlines added** | `f.writelines(x + "\n" for x in xs)` |
| `str(x)` / f-strings | Numbers must become text | `f.write(f"{score}")` |
| `"\n"` | You provide all line breaks | `f.write(item + "\n")` |
| `with ... as f` | Flush + close on every exit path | `with open(p, "w") as f:` |
| `Path.write_text()` | One-line write shortcut | `Path(p).write_text(t, encoding="utf-8")` |

Key takeaways:

- `"w"` = blank page: creates missing files, obliterates existing ones, no questions asked.
- `.write()` writes exactly the characters you give it — conversions and newlines are your responsibility.
- `writelines()` is "write each", not "write lines".
- Structure data as lines-in-a-list, join once, write once — and always inside `with`.

## 🔗 Next Lesson

Sometimes you want to *keep* the old file and just add to it — logs, diaries, histories. That is append mode, plus the strict `"x"` mode that refuses to overwrite: [`../03_Append_File/notes.ipynb`](../03_Append_File/notes.ipynb).